### 📊 Experiments Results

🔹 1. Imports

In [126]:
import json
import pandas as pd
from pathlib import Path

🔹 2. Função de interpretação

In [127]:
def interpret(value, metric):
    if metric == "accuracy":
        if value > 0.8: return "🔥 Muito bom"
        elif value > 0.6: return "👍 Ok"
        else: return "⚠️ Baixo"
    
    if metric == "proxy_score":
        if value > 0.75: return "🔥 Forte"
        elif value > 0.5: return "👍 Médio"
        else: return "⚠️ Fraco"

🔹 3. Load + processamento

In [128]:
def load_and_process(path):
    with open(path, "r") as f:
        data = json.load(f)

    rows = []

    for item in data:
        pred = item.get("prediction").upper()
        gold = item.get("gold_label").upper()

        steps = item.get("steps", [])
        evidence_counts = [len(s.get("evidence", [])) for s in steps]

        avg_evidence = sum(evidence_counts) / len(evidence_counts) if evidence_counts else 0

        rows.append({
            "claim_id": item.get("id") or item.get("claim"),
            "prediction": pred,
            "label": gold,
            "correct": pred == gold,
            "num_steps": len(steps),
            "avg_evidence_per_step": avg_evidence,
        })

    df = pd.DataFrame(rows)

    # métricas
    accuracy = df["correct"].mean()
    evidence_score = df["avg_evidence_per_step"].mean()
    proxy_score = 0.7 * accuracy + 0.3 * min(evidence_score / 5.0, 1.0)

    metrics = {
        "accuracy": accuracy,
        "proxy_score": proxy_score
    }

    return df, metrics

🔹 4. Paths

In [129]:
paths = {
    "baseline": [
        "../outputs/averitec_original_pipeline/2026-03-24_19-33-17/averitec_dev.json"
    ],
    "iterative": [
        "../outputs/averitec_iterative_pipeline/2026-03-24_19-41-59/averitec_dev.json"
    ]
}

🔹 5. Carregamento geral

In [130]:
dfs = {}
results = {}

for pipeline, file_list in paths.items():
    dfs[pipeline] = []
    results[pipeline] = []

    for path in file_list:
        df, metrics = load_and_process(path)

        run_name = Path(path).parent.name
        df["run"] = run_name

        dfs[pipeline].append(df)
        results[pipeline].append(metrics)

🔹 6. Concatenar tudo

In [131]:
dfs_concat = {
    k: pd.concat(v, ignore_index=True)
    for k, v in dfs.items()
}

🔹 7. Print por pipeline

In [ ]:
from pathlib import Path
from sklearn.metrics import classification_report

import warnings
from sklearn.exceptions import UndefinedMetricWarning
warnings.filterwarnings("ignore", category=UndefinedMetricWarning)

def print_pipeline_summary(name, df):
    run_name = Path(path).parent.name
    dataset_name = Path(path).name

    df["run"] = run_name
    df["dataset"] = dataset_name

    for run, df_run in df.groupby("run"):

        accuracy = df_run["correct"].mean()
        evidence_score = df_run["avg_evidence_per_step"].mean()
        

        df_errors = df_run[~df_run["correct"]]

        print("=" * 50)
        print(f"📌 PIPELINE: {name.upper()} - {run}")
        #print(f"📂 ARQUIVO: {run}")
        print(f"📂 DATASET: {df_run['dataset'].iloc[0]}")
        print("\n")
        print('🎯 Accuracy:', round(accuracy, 3), '->', interpret(accuracy, 'accuracy'))
        print("\n")

        print("📊 Total samples:", len(df_run))
        print("✅ Correct predictions:", df_run["correct"].sum())
        print("❌ Errors:", len(df_errors))
        print("=" * 50)
        #print(classification_report(df["label"], df["prediction"]))
        #print("📊 Per-class accuracy:")
        #print(df.groupby("label")["correct"].mean())

for name in dfs_concat:
    print_pipeline_summary(name, dfs_concat[name])

📌 PIPELINE: BASELINE - 2026-03-24_19-41-59
📂 DATASET: averitec_dev.json


🎯 Accuracy: 0.564 -> ⚠️ Baixo


📊 Total samples: 500
✅ Correct predictions: 282
❌ Errors: 218
                                    precision    recall  f1-score   support

CONFLICTING EVIDENCE/CHERRYPICKING       0.00      0.00      0.00        38
               NOT ENOUGH EVIDENCE       0.11      0.34      0.17        35
                           REFUTED       0.70      0.75      0.72       305
                         SUPPORTED       0.65      0.34      0.44       122

                          accuracy                           0.56       500
                         macro avg       0.36      0.36      0.33       500
                      weighted avg       0.59      0.56      0.56       500

📊 Per-class accuracy:
label
CONFLICTING EVIDENCE/CHERRYPICKING    0.000000
NOT ENOUGH EVIDENCE                   0.342857
REFUTED                               0.750820
SUPPORTED                             0.336066
Name:

In [133]:

paths = {
    "baseline": [
        "../outputs/averitec_original_pipeline/2026-03-24_19-33-17/averitec_train.json"
    ],
    "iterative": [
        "../outputs/averitec_iterative_pipeline/2026-03-24_19-41-59/averitec_train.json"
    ]
}

dfs = {}
results = {}

for pipeline, file_list in paths.items():
    dfs[pipeline] = []
    results[pipeline] = []

    for path in file_list:
        df, metrics = load_and_process(path)

        run_name = Path(path).parent.name
        df["run"] = run_name

        dfs[pipeline].append(df)
        results[pipeline].append(metrics)

dfs_concat = {
    k: pd.concat(v, ignore_index=True)
    for k, v in dfs.items()
}

for name in dfs_concat:
    print_pipeline_summary(name, dfs_concat[name])

📌 PIPELINE: BASELINE - 2026-03-24_19-41-59
📂 DATASET: averitec_train.json


🎯 Accuracy: 0.536 -> ⚠️ Baixo


📊 Total samples: 481
✅ Correct predictions: 258
❌ Errors: 223
                                    precision    recall  f1-score   support

CONFLICTING EVIDENCE/CHERRYPICKING       0.00      0.00      0.00        30
               NOT ENOUGH EVIDENCE       0.12      0.39      0.19        33
                           REFUTED       0.68      0.72      0.70       283
                         SUPPORTED       0.56      0.31      0.40       135

                          accuracy                           0.54       481
                         macro avg       0.34      0.36      0.32       481
                      weighted avg       0.57      0.54      0.54       481

📊 Per-class accuracy:
label
CONFLICTING EVIDENCE/CHERRYPICKING    0.000000
NOT ENOUGH EVIDENCE                   0.393939
REFUTED                               0.717314
SUPPORTED                             0.311111
Nam